# Human Approval & Permissions
Enterprise agents cannot be given unfettered access to destructive actions (e.g., executing a refund, deleting a database). SOTA architectures require **Human-in-the-Loop (HITL)** choke points and **Idempotency Keys** to prevent duplicate actions during network retries.

This notebook demonstrates these concepts using the `langgraph` SDK.

**Dependencies required:** `pip install langgraph pydantic`


## 1. Idempotency Keys (Surviving Network Retries)
If an agent executes an action, but the HTTP response times out, the agent might retry the action. Without idempotency, the user gets double-charged.


In [ ]:
from pydantic import BaseModel, Field
import uuid

# ❌ DANGEROUS TOOL
def process_refund_unsafe(user_id: int, amount: float):
    # If the network fails and the LLM retries, the user gets refunded twice!
    print(f"   [API] Processing refund of ${amount} for User {user_id}...")
    return "Refund processed."

# ✅ SOTA TOOL WITH IDEMPOTENCY
db_processed_keys = set() # Mock database of processed keys

class RefundInput(BaseModel):
    user_id: int = Field(description="The ID of the user.")
    amount: float = Field(description="The refund amount.")
    idempotency_key: str = Field(description="A unique UUID v4 generated by the agent for this specific transaction attempt.")

def process_refund_safe(user_id: int, amount: float, idempotency_key: str):
    if idempotency_key in db_processed_keys:
        print(f"   [API] Ignoring duplicate request. Key {idempotency_key} already processed.")
        return "Refund already processed (Idempotent success)."
        
    print(f"   [API] Processing new refund of ${amount} for User {user_id}...")
    db_processed_keys.add(idempotency_key)
    return "Refund processed successfully."

print("❌ Unsafe Execution (Retry Simulation):")
process_refund_unsafe(991, 50.0)
process_refund_unsafe(991, 50.0) # Double charged!

print("\n✅ Safe Execution (Retry Simulation):")
# The LLM generates ONE key when deciding to refund. It uses the same key on retry.
transaction_key = str(uuid.uuid4()) 
process_refund_safe(991, 50.0, transaction_key)
process_refund_safe(991, 50.0, transaction_key) # Blocked!


❌ Unsafe Execution (Retry Simulation):
   [API] Processing refund of $50.0 for User 991...
   [API] Processing refund of $50.0 for User 991...

✅ Safe Execution (Retry Simulation):
   [API] Processing new refund of $50.0 for User 991...
   [API] Ignoring duplicate request. Key e41c8f8b-f421-4f16-a111-d1469e710000 already processed.


## 2. Human-in-the-Loop (HITL) using LangGraph
We use LangGraph's checkpointer to pause execution *before* a dangerous node executes.


In [ ]:
import sqlite3
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.sqlite import SqliteSaver

# 1. Define State
class AgentState(TypedDict):
    action_plan: str
    amount: float
    is_approved: bool

# 2. Define Nodes
def planning_node(state: AgentState):
    print("🤖 [Node: Planner] Decided to refund user $500.")
    return {"action_plan": "Execute refund", "amount": 500.0, "is_approved": False}

def dangerous_action_node(state: AgentState):
    # This node actually touches production data
    print(f"💸 [Node: Executor] Executing refund of ${state['amount']}.")
    return {"action_plan": "Refund complete."}

# 3. Build Graph
workflow = StateGraph(AgentState)
workflow.add_node("planner", planning_node)
workflow.add_node("execute_action", dangerous_action_node)

workflow.set_entry_point("planner")
workflow.add_edge("planner", "execute_action")
workflow.add_edge("execute_action", END)

# 4. Attach Memory & Explicit Breakpoints
conn = sqlite3.connect(":memory:", check_same_thread=False)
memory = SqliteSaver(conn)

# SOTA FEATURE: We tell LangGraph to ALWAYS pause BEFORE entering "execute_action"
app = workflow.compile(
    checkpointer=memory,
    interrupt_before=["execute_action"] 
)
print("✅ LangGraph compiled with an Explicit Breakpoint.")


✅ LangGraph compiled with an Explicit Breakpoint.


## 3. Simulating the Approval Flow
Watch how the graph halts execution, waits for an out-of-band state update (the human clicking 'Approve'), and then resumes.


In [ ]:
thread_config = {"configurable": {"thread_id": "ticket_001"}}

print("--- RUN 1: AI Plans the Action ---")
# The graph runs, hits the breakpoint, and halts cleanly.
app.invoke({"action_plan": "", "amount": 0.0, "is_approved": False}, config=thread_config)

print("\n[SYSTEM] Graph execution paused. Waiting for human...")

print("\n--- OUT-OF-BAND: Human Approval UI ---")
# An engineer reviews the checkpoint in a dashboard and approves it
app.update_state(thread_config, {"is_approved": True})
print("👨‍💻 Human clicked 'Approve'. State updated.")

print("\n--- RUN 2: Resuming Execution ---")
# Invoking with 'None' tells LangGraph to resume from the breakpoint
app.invoke(None, config=thread_config)


--- RUN 1: AI Plans the Action ---
🤖 [Node: Planner] Decided to refund user $500.

[SYSTEM] Graph execution paused. Waiting for human...

--- OUT-OF-BAND: Human Approval UI ---
👨‍💻 Human clicked 'Approve'. State updated.

--- RUN 2: Resuming Execution ---
💸 [Node: Executor] Executing refund of $500.0.
